# 05 - Modele de Serie Temporelle : Evolution de Performance des Joueurs

Ce notebook construit le modele derriere le graphique **"Evolution de performance"**
de la page `/joueurs/performances` du site.

**Objectifs**
- Recuperer les donnees **StatsBomb Open Data** (La Liga 2020/2021) via `statsbombpy`.
- Agreger les evenements en un **score de performance (0-100)** par joueur et par match.
- Construire une **serie temporelle mensuelle** par joueur (indice de forme lisse).
- Entrainer un **modele de serie temporelle** (regression par features de decalage / *lags*)
  avec XGBoost pour prevoir la performance du mois suivant.
- Viser une **precision > 0.96** (R2 en prevision a 1 pas + accuracy a tolerance +/- 3 pts).
- Exporter le modele dans `../models/player_performance_model.joblib` pour la suite (FastAPI + Streamlit).

> Le notebook fonctionne meme hors-ligne : si StatsBomb est indisponible, un **jeu synthetique
> realiste** (series AR(1) lissees) prend le relais. La config est surchargeable par variables
> d'environnement (`PPTS_*`) pour les tests automatises.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Configuration (surchargeable via variables d'environnement PPTS_*) ---
COMPETITION_ID = int(os.getenv("PPTS_COMPETITION_ID", "11"))   # 11 = La Liga
SEASON_ID      = int(os.getenv("PPTS_SEASON_ID", "27"))        # 27 = 2020/2021
_mm            = os.getenv("PPTS_MAX_MATCHES", "all")          # saison complete par defaut
MAX_MATCHES    = None if str(_mm).lower() in ("none", "all", "0") else int(_mm)
MIN_MONTHS     = int(os.getenv("PPTS_MIN_MONTHS", "8"))        # historique suffisant pour une prevision fiable
FORCE_SYNTHETIC = os.getenv("PPTS_FORCE_SYNTHETIC", "0") == "1"

LAGS         = [1, 2, 3]        # decalages utilises comme features
ROLL_WINDOW  = 3               # fenetre glissante
EWMA_SPAN    = 3               # lissage de l'indice de forme (comme la courbe du site)
SCORE_RANGE  = (60.0, 99.0)    # plage cible du score (axe Y du graphique du site)
TOL_PTS      = 3.0             # tolerance pour l'accuracy

# Works both in VS Code (workspace cwd) and Jupyter (notebooks cwd).
ROLE_PLAYER_DIR = os.path.abspath(os.path.join(os.getcwd(), "ml_role_player"))
if not os.path.isdir(ROLE_PLAYER_DIR):
    ROLE_PLAYER_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
MODELS_DIR = os.path.join(ROLE_PLAYER_DIR, "models")
MODEL_PATH = os.path.join(MODELS_DIR, "player_performance_model.joblib")

print("Config :",
      f"comp={COMPETITION_ID} season={SEASON_ID} max_matches={MAX_MATCHES}",
      f"min_months={MIN_MONTHS} force_synthetic={FORCE_SYNTHETIC}")

## 1. Collecte des donnees StatsBomb

On telecharge les matchs de la saison puis, pour chaque match, on agrege les evenements
en statistiques **par joueur** (passes, tirs, buts, xG, passes cles, dribbles, actions defensives...).
On agrege match par match pour garder une empreinte memoire faible.

In [ ]:
def aggregate_match(events, match_id, match_date):
    # Agrege les evenements d'un match en une ligne de stats par joueur.
    rows = []
    if events is None or len(events) == 0 or "player" not in events.columns:
        return rows
    ev = events[events["player"].notna()].copy()

    def col(name):
        return ev[name] if name in ev.columns else pd.Series([np.nan] * len(ev), index=ev.index)

    ev["_type"]         = col("type")
    ev["_pass_outcome"] = col("pass_outcome")
    ev["_shot_outcome"] = col("shot_outcome")
    ev["_xg"]           = pd.to_numeric(col("shot_statsbomb_xg"), errors="coerce")
    ev["_dribble_out"]  = col("dribble_outcome")
    ev["_shot_assist"]  = col("pass_shot_assist")
    ev["_goal_assist"]  = col("pass_goal_assist")

    for player, g in ev.groupby("player"):
        team = g["team"].iloc[0] if "team" in g.columns else None
        passes = int((g["_type"] == "Pass").sum())
        passes_ok = int(((g["_type"] == "Pass") & (g["_pass_outcome"].isna())).sum())
        comp_rate = (passes_ok / passes) if passes > 0 else np.nan
        dribbles = int((g["_type"] == "Dribble").sum())
        dribbles_ok = int(((g["_type"] == "Dribble") & (g["_dribble_out"] == "Complete")).sum())
        dribble_rate = (dribbles_ok / dribbles) if dribbles > 0 else np.nan
        rows.append({
            "player": player,
            "team": team,
            "match_id": match_id,
            "match_date": match_date,
            "n_events": int(len(g)),
            "passes": passes,
            "comp_rate": comp_rate,
            "shots": int((g["_type"] == "Shot").sum()),
            "goals": int(((g["_type"] == "Shot") & (g["_shot_outcome"] == "Goal")).sum()),
            "xg": float(g["_xg"].fillna(0).sum()),
            "key_passes": int(g["_shot_assist"].fillna(False).astype(bool).sum()),
            "assists": int(g["_goal_assist"].fillna(False).astype(bool).sum()),
            "dribbles": dribbles,
            "dribble_rate": dribble_rate,
            "defensive": int(g["_type"].isin(
                ["Interception", "Ball Recovery", "Block", "Clearance", "Duel", "Foul Won"]).sum()),
        })
    return rows


df_match_player = pd.DataFrame()
data_source = "synthetic"

if not FORCE_SYNTHETIC:
    try:
        from statsbombpy import sb
        _ = sb.competitions()
        matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
        matches = matches.sort_values("match_date").reset_index(drop=True)
        if MAX_MATCHES is not None:
            matches = matches.head(MAX_MATCHES)
        print(f"Competition {COMPETITION_ID} / saison {SEASON_ID} : {len(matches)} matchs a traiter.")

        all_rows = []
        n = len(matches)
        for i, m in enumerate(matches.itertuples(index=False), start=1):
            try:
                ev = sb.events(match_id=m.match_id)
                all_rows.extend(aggregate_match(ev, m.match_id, m.match_date))
            except Exception as e:
                print(f"  [skip] match {m.match_id}: {e}")
            if i % 20 == 0 or i == n:
                print(f"  {i}/{n} matchs traites -> {len(all_rows)} lignes joueur-match.")

        df_match_player = pd.DataFrame(all_rows)
        if len(df_match_player) > 0:
            data_source = "statsbomb"
        else:
            print("Aucune donnee recuperee -> bascule synthetique.")
    except Exception as e:
        print(f"[WARN] StatsBomb indisponible ({type(e).__name__}: {e}). Bascule synthetique.")

print("Source de donnees:", data_source, "| lignes joueur-match:", len(df_match_player))

## 2. Score de performance & agregation mensuelle

Chaque match est resume en un **score composite (0-100)** : les composantes (buts, passes decisives,
xG, passes cles, taux de reussite des passes/dribbles, actions defensives, implication) sont
standardisees (z-score) puis combinees et projetees dans la plage du graphique via une logistique.

On agrege ensuite par **mois** (moyenne), puis on calcule un **indice de forme lisse** (EWMA) :
c'est exactement le type de courbe montree sur le site.

Un generateur **synthetique** (series AR(1) lissees et autocorrelees) sert de repli.

In [ ]:
def compute_perf_scores(dfmp, score_range=SCORE_RANGE):
    # Score composite (0-100) par joueur-match.
    df = dfmp.copy()
    df["comp_rate"] = df["comp_rate"].fillna(df["comp_rate"].median())
    df["dribble_rate"] = df["dribble_rate"].fillna(df["dribble_rate"].median())

    weights = {
        "goals": 1.6, "assists": 1.3, "xg": 1.1, "shots": 0.4, "key_passes": 0.7,
        "comp_rate": 1.0, "dribble_rate": 0.5, "defensive": 0.5, "n_events": 0.8,
    }
    z = pd.DataFrame(index=df.index)
    for c in weights:
        v = df[c].astype(float)
        std = v.std(ddof=0)
        z[c] = 0.0 if (std == 0 or np.isnan(std)) else (v - v.mean()) / std

    composite = sum(weights[c] * z[c] for c in weights)
    cz_std = composite.std(ddof=0)
    cz = composite / (cz_std if cz_std else 1.0)
    lo, hi = score_range
    df["perf_match"] = lo + (hi - lo) * (1.0 / (1.0 + np.exp(-cz)))
    return df


def to_monthly(df):
    # Agrege les scores match en points mensuels par joueur.
    df = df.copy()
    df["match_date"] = pd.to_datetime(df["match_date"])
    df["period"] = df["match_date"].dt.to_period("M").dt.to_timestamp()
    g = (df.groupby(["player", "period"])
           .agg(perf_score=("perf_match", "mean"),
                matches_played=("match_id", "nunique"))
           .reset_index()
           .sort_values(["player", "period"]))
    g["month_index"] = g.groupby("player").cumcount()
    return g


def synthetic_monthly(n_players=240, n_months=9, seed=RANDOM_STATE, score_range=SCORE_RANGE):
    # Panel mensuel synthetique : niveau latent par joueur + trajectoire AR(1) lissee.
    rng = np.random.default_rng(seed)
    lo, hi = score_range
    base_date = pd.Timestamp("2020-09-01")
    rows = []
    for p in range(n_players):
        ability = rng.normal(0, 1.1)
        level = lo + (hi - lo) * (1.0 / (1.0 + np.exp(-ability)))
        trend = rng.normal(0, 0.5)
        x = level + rng.normal(0, 1.5)
        months = int(rng.integers(MIN_MONTHS, n_months + 1))
        for k in range(months):
            x = 0.75 * x + 0.25 * (level + trend * k) + rng.normal(0, 0.8)
            rows.append({
                "player": f"Player_{p:03d}",
                "period": base_date + pd.DateOffset(months=k),
                "perf_score": float(np.clip(x, lo, hi)),
                "matches_played": int(rng.integers(2, 5)),
            })
    g = pd.DataFrame(rows).sort_values(["player", "period"])
    g["month_index"] = g.groupby("player").cumcount()
    return g

In [ ]:
# Construction de la table mensuelle (reelle si dispo, sinon synthetique)
if data_source == "statsbomb":
    df_scored = compute_perf_scores(df_match_player)
    df_monthly = to_monthly(df_scored)
else:
    df_monthly = synthetic_monthly()

# On ne garde que les joueurs avec assez de mois actifs (pour construire les lags)
counts = df_monthly.groupby("player")["period"].transform("count")
df_monthly = df_monthly[counts >= MIN_MONTHS].reset_index(drop=True)

# Repli si trop peu de joueurs reels qualifies (ex: petit echantillon de matchs)
if df_monthly["player"].nunique() < 30:
    print(f"[info] Seulement {df_monthly['player'].nunique()} joueurs qualifies -> repli synthetique.")
    data_source = "synthetic"
    df_monthly = synthetic_monthly()
    counts = df_monthly.groupby("player")["period"].transform("count")
    df_monthly = df_monthly[counts >= MIN_MONTHS].reset_index(drop=True)

# Indice de forme lisse (EWMA) : la courbe "Evolution de performance" du site
df_monthly["perf_form"] = (df_monthly.groupby("player")["perf_score"]
                           .transform(lambda s: s.ewm(span=EWMA_SPAN, adjust=False).mean()))

print("Source finale :", data_source)
print("Joueurs modelises :", df_monthly["player"].nunique(), "| points mensuels :", len(df_monthly))
df_monthly.head(10)

## 3. Visualisation "Evolution de performance"

Reproduction de la courbe du site pour un joueur representatif (indice de forme lisse).

In [ ]:
import matplotlib.pyplot as plt

FR_MONTHS = {1: "Jan", 2: "Fev", 3: "Mar", 4: "Avr", 5: "Mai", 6: "Juin",
             7: "Juil", 8: "Aou", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}

sample_player = df_monthly["player"].value_counts().index[0]
s = df_monthly[df_monthly["player"] == sample_player].sort_values("period")

fig, ax = plt.subplots(figsize=(8.2, 4.3))
fig.patch.set_facecolor("#0f1420")
ax.set_facecolor("#0f1420")
ax.plot(range(len(s)), s["perf_form"], marker="o", color="#ff5a4d",
        linewidth=2.4, markersize=7, markeredgecolor="#ff5a4d")
ax.set_ylim(60, 100)
ax.set_title(f"Evolution de performance  -  {sample_player}", color="white",
             fontsize=13, loc="left", fontweight="bold", pad=14)
ax.set_xticks(range(len(s)))
ax.set_xticklabels([FR_MONTHS[d.month] for d in s["period"]], color="#9aa4b2")
ax.tick_params(colors="#9aa4b2")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.grid(True, color="#2a3346", linestyle="--", linewidth=0.6, alpha=0.7)
plt.tight_layout()
plt.show()

## 4. Feature engineering temporel (lags & fenetres glissantes)

On transforme la serie en probleme supervise : pour chaque (joueur, mois) on construit des
features **strictement passees** (decalages, moyenne/ecart-type glissants, moyenne cumulee, variation).
La cible est l'indice de forme du mois courant -> prevision a 1 pas.

In [ ]:
def make_supervised(df, lags=LAGS, roll=ROLL_WINDOW, value_col="perf_form"):
    df = df.sort_values(["player", "period"]).copy()

    def per_player(g):
        g = g.copy()
        s = g[value_col]
        for L in lags:
            g[f"lag_{L}"] = s.shift(L)
        past = s.shift(1)  # decale de 1 -> pas de fuite d'information
        g["roll_mean"] = past.rolling(roll, min_periods=1).mean()
        g["roll_std"] = past.rolling(roll, min_periods=2).std()
        g["expanding_mean"] = past.expanding(min_periods=1).mean()
        return g

    df = df.groupby("player", group_keys=False).apply(per_player)
    df["delta_prev"] = df["lag_1"] - df["lag_2"]
    df["roll_std"] = df["roll_std"].fillna(0.0)
    df = df.dropna(subset=[f"lag_{L}" for L in lags]).reset_index(drop=True)
    return df


sup = make_supervised(df_monthly)
FEATURES = ([f"lag_{L}" for L in LAGS]
            + ["roll_mean", "roll_std", "expanding_mean", "delta_prev",
               "month_index", "matches_played"])
TARGET = "perf_form"

print("Echantillons supervises :", len(sup), "| nb features :", len(FEATURES))
print("Features :", FEATURES)
sup[FEATURES + [TARGET]].head()

## 5. Split temporel & entrainement (XGBoost)

Validation honnete pour une serie temporelle : on **retient le dernier mois de chaque joueur
comme test** (prevision a 1 pas) et on entraine sur le passe. Un arbre n'a pas besoin de scaling.

In [ ]:
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sup = sup.sort_values(["player", "period"]).reset_index(drop=True)
is_last = sup.groupby("player")["period"].transform("max") == sup["period"]
train, test = sup[~is_last], sup[is_last]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f"Entrainement : {len(X_train)} lignes | Test (dernier mois/joueur) : {len(X_test)} lignes")

model = xgb.XGBRegressor(
    n_estimators=600, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
    reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=-1,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

# Baseline de reference : persistance (prevision = valeur du mois precedent)
baseline_pred = X_test["lag_1"].values
print(f"Baseline persistance  R2 = {r2_score(y_test, baseline_pred):.4f}")

## 6. Evaluation

- **R2** (prevision a 1 pas) = "precision" du modele de regression temporelle -> objectif > 0.96.
- **Accuracy a tolerance** = part des previsions a moins de +/- 3 points de la valeur reelle.
- On reporte aussi le R2 vis-a-vis du score mensuel **brut** (non lisse) par transparence.

In [ ]:
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = float(mean_squared_error(y_test, pred) ** 0.5)
tol_acc = float(np.mean(np.abs(pred - y_test.values) <= TOL_PTS))
r2_raw = r2_score(test["perf_score"].values, pred)  # vs score brut (plus difficile)

print(f"R2 (indice de forme)      : {r2:.4f}")
print(f"MAE                       : {mae:.3f} pts")
print(f"RMSE                      : {rmse:.3f} pts")
print(f"Accuracy (|err| <= {TOL_PTS} pts) : {tol_acc:.4f}")
print(f"R2 vs score brut          : {r2_raw:.4f}")
print()
status = "ATTEINT ✅" if (r2 > 0.96 and tol_acc > 0.96) else "NON ATTEINT ⚠️"
print(f"Objectif precision > 0.96 : {status}")

# Nuage previsions vs realite
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(y_test, pred, alpha=0.6, color="#ff5a4d", edgecolor="none", s=28)
lims = [min(y_test.min(), pred.min()) - 1, max(y_test.max(), pred.max()) + 1]
ax.plot(lims, lims, "--", color="#4a5568", linewidth=1)
ax.set_xlabel("Reel"); ax.set_ylabel("Prevu")
ax.set_title(f"Previsions vs realite (R2 = {r2:.3f})")
ax.set_xlim(lims); ax.set_ylim(lims)
plt.tight_layout(); plt.show()

# Importance des features
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
imp.plot(kind="barh", ax=ax, color="#ff5a4d")
ax.set_title("Importance des features"); plt.tight_layout(); plt.show()

## 7. Prevision recursive (demo)

Prevision des prochains mois pour un joueur : on reinjecte chaque prevision pour predire la suivante.
C'est la fonction que le service FastAPI reutilisera.

In [ ]:
def forecast_player(model, history, steps=3, features=FEATURES,
                    lags=LAGS, roll=ROLL_WINDOW, start_month_index=None,
                    matches_played=3):
    # history : liste des valeurs de forme passees (perf_form) ; retourne `steps` previsions.
    scores = list(history)
    preds = []
    mi = (len(scores) - 1) if start_month_index is None else start_month_index
    for _ in range(steps):
        row = {}
        for L in lags:
            row[f"lag_{L}"] = scores[-L] if len(scores) >= L else scores[0]
        window = scores[-roll:]
        row["roll_mean"] = float(np.mean(window))
        row["roll_std"] = float(np.std(window)) if len(window) >= 2 else 0.0
        row["expanding_mean"] = float(np.mean(scores))
        row["delta_prev"] = (scores[-1] - scores[-2]) if len(scores) >= 2 else 0.0
        mi += 1
        row["month_index"] = mi
        row["matches_played"] = matches_played
        yhat = float(model.predict(pd.DataFrame([row])[features])[0])
        preds.append(yhat)
        scores.append(yhat)
    return preds


hist = df_monthly[df_monthly["player"] == sample_player].sort_values("period")["perf_form"].tolist()
future = forecast_player(model, hist, steps=3)
print(f"Historique {sample_player} :", [round(v, 1) for v in hist])
print("Prevision 3 mois           :", [round(v, 1) for v in future])

fig, ax = plt.subplots(figsize=(8.2, 4.3))
fig.patch.set_facecolor("#0f1420"); ax.set_facecolor("#0f1420")
xs_hist = list(range(len(hist)))
xs_fut = list(range(len(hist) - 1, len(hist) + len(future)))
ax.plot(xs_hist, hist, marker="o", color="#ff5a4d", linewidth=2.4, markersize=7, label="Historique")
ax.plot(xs_fut, [hist[-1]] + future, marker="o", linestyle="--", color="#4dd0e1",
        linewidth=2.2, markersize=7, label="Prevision")
ax.set_ylim(60, 100)
ax.set_title(f"Prevision de performance  -  {sample_player}", color="white",
             fontsize=13, loc="left", fontweight="bold", pad=14)
ax.tick_params(colors="#9aa4b2")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.grid(True, color="#2a3346", linestyle="--", linewidth=0.6, alpha=0.7)
ax.legend(facecolor="#0f1420", edgecolor="#2a3346", labelcolor="white")
plt.tight_layout(); plt.show()

## 8. Export du modele

Sauvegarde dans `../models/player_performance_model.joblib` au format dictionnaire (comme les autres
modeles du projet), avec les metriques et toutes les meta-donnees necessaires au deploiement FastAPI.

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

sample_series = (df_monthly[df_monthly["player"] == sample_player][["period", "perf_score", "perf_form"]]
                 .assign(period=lambda d: d["period"].dt.strftime("%Y-%m"))
                 .to_dict("records"))

artifact = {
    "model": model,
    "model_type": "XGBRegressor",
    "task": "player_performance_timeseries_forecast",
    "features": FEATURES,
    "target": TARGET,
    "lags": LAGS,
    "roll_window": ROLL_WINDOW,
    "ewma_span": EWMA_SPAN,
    "score_range": SCORE_RANGE,
    "min_months": MIN_MONTHS,
    "scaler": None,  # modele a base d'arbres -> pas de scaling requis
    "data_source": data_source,
    "metrics": {
        "r2": float(r2),
        "mae": float(mae),
        "rmse": float(rmse),
        "tolerance_accuracy": float(tol_acc),
        "tolerance_pts": float(TOL_PTS),
        "r2_vs_raw_score": float(r2_raw),
    },
    "accuracy_score": float(r2),  # coherent avec la convention des autres notebooks
    "sample_player": str(sample_player),
    "sample_series": sample_series,
    "created_with": "05_player_performance_timeseries.ipynb",
}

joblib.dump(artifact, MODEL_PATH)
print("Modele exporte ->", os.path.abspath(MODEL_PATH))
print("Taille :", round(os.path.getsize(MODEL_PATH) / 1024, 1), "Ko")
print("Metriques :", artifact["metrics"])

## 9. Prochaines etapes

1. **FastAPI** : ajouter un endpoint (ex: `/predict-performance`) qui charge `player_performance_model.joblib`,
   recoit l'historique de forme d'un joueur et renvoie les previsions via `forecast_player(...)`.
2. **Streamlit** : petite page de test qui appelle l'API et affiche la courbe "Evolution de performance".
3. Brancher sur la page `/joueurs/performances` du front (plus tard).

> Le modele et les fonctions `make_supervised` / `forecast_player` ci-dessus sont volontairement
> autonomes pour etre reutilises tels quels cote service.